# Q3 baseline: the co-author seed and first journal entry

The question

- does having a co-author who already published in a journal raise the chance of entering that journal for the first time

The unit is one opportunity, a row per (author, journal, year) where the author was active and had not entered the journal yet, from the event table built in `logic/build_event_table.py`. `C` is the seed, an earlier collaborator who was in the journal before year t. `F` is first entry in year t. Q3 proper is the topic-adjusted ratio, so everything here splits in two

1. the naive contrast `F ~ C`, computable today and explicitly unadjusted
2. a provisional adjustment `F ~ C + T` with Pierre's rolling topic fit, marked provisional because the export is a local run pending Pierre's confirmation, the treatment of missing `T` is open, and the frozen-profile sensitivity has no common anchor for `C = 0`

The final numbers come after the threshold-free export is approved and the missing-`T` decision. The independent row-by-row comparison against the Prolog implementation matched all 6,422,558 rows; review of those rules is a separate validation track, not a Q3 blocker. Variant A is the main opportunity set and B the sensitivity, so this notebook reads variant A only.

Inputs, both local in `../data/` and not in git

- `event_table_python_v0_oppA.csv`, 6,422,558 rows
- `event_table_topicmatch_local_nothreshold.csv`, the same rows with `topic_match` added. This is a local execution of the pipeline in PR #58 with the whole-period paper threshold removed, as proposed there. The port reproduces the committed export exactly under the original settings (max deviation 2e-7 on all 623,510 values), so the only change is the one intended. Pierre's own rerun replaces it when it exists

## Step 1: load and align

- both files carry the same rows in the same order, which is asserted on the key columns before the topic column is attached
- `F` is read off `entering_work_id`, filled exactly on entry rows

In [1]:
import numpy as np, pandas as pd

usecols = ["author_id", "journal_id", "t", "n_prior_papers", "coauthor_seed",
           "first_entry_ride", "entering_work_id"]
ev = pd.read_csv("../data/event_table_python_v0_oppA.csv", usecols=usecols, low_memory=False,
                 dtype={"t": "int16", "n_prior_papers": "int32", "coauthor_seed": "int8",
                        "first_entry_ride": "int8", "entering_work_id": "str"})
tm = pd.read_csv("../data/event_table_topicmatch_local_nothreshold.csv", low_memory=False,
                 usecols=["author_id", "journal_id", "t", "topic_match", "tm_status", "profile_cutoff"],
                 dtype={"t": "int16"})

# same rows in the same order, otherwise attaching by position would silently mix rows
assert len(ev) == len(tm)
assert (ev["author_id"].values == tm["author_id"].values).all()
assert (ev["journal_id"].values == tm["journal_id"].values).all()
assert (ev["t"].values == tm["t"].values).all()
ev["T"] = tm["topic_match"].values
ev["tm_status"] = tm["tm_status"].values

# the status column must explain the fill exactly, and every profile must stop before t
assert ((tm["tm_status"] == "ok") == tm["topic_match"].notna()).all()
assert (tm.loc[tm.tm_status == "ok", "profile_cutoff"] < tm.loc[tm.tm_status == "ok", "t"]).all()
del tm

ev["F"] = ev["entering_work_id"].notna().astype("int8")
C = ev["coauthor_seed"].values
F = ev["F"].values
print(f"{len(ev):,} rows, {int(F.sum()):,} entries, {(C==1).sum():,} rows with a seed")

6,422,558 rows, 96,819 entries, 19,035 rows with a seed


## Step 2: check against the build

The event table build printed its own headline counts. Rebuilding them here from the loaded frame confirms nothing was lost or doubled on the way in.

In [2]:
assert int(F.sum()) == 96819                       # one entry row per (author, journal) pair in the corpus
assert int((C == 1).sum()) == 19035                # rows where the seed predates t
assert int(F[C == 1].sum()) == 1784                # entries that happened with a seed in place
assert int(ev["first_entry_ride"].sum()) == 756    # entries with the qualifying seed co-author on the paper
# T coverage changes between export versions, the fill contract is asserted at load time instead
print(f"all four event counts match the build, T exists on {ev['T'].notna().sum():,} rows in this export")

all four event counts match the build, T exists on 1,106,356 rows in this export


## Step 3: the naive contrast

- entry rate with a seed against entry rate without, nothing held constant
- this number is confounded by design, an author with a seed is better connected and probably closer to the journal's topics, so read it as a crude association, not an effect in either direction
- the interval resamples whole authors, because one author contributes many related rows and row-level intervals would be too narrow

In [3]:
p1, p0 = F[C == 1].mean(), F[C == 0].mean()
print(f"entry rate with a seed    {p1:.4%}  ({int(F[C==1].sum()):,} of {(C==1).sum():,})")
print(f"entry rate without        {p0:.4%}  ({int(F[C==0].sum()):,} of {(C==0).sum():,})")
print(f"naive rate ratio          {p1/p0:.2f}")

# author level counts once, then the bootstrap only touches these four arrays
codes, authors = pd.factorize(ev["author_id"])
nA = len(authors)
n1 = np.bincount(codes[C == 1], minlength=nA)
e1 = np.bincount(codes[(C == 1) & (F == 1)], minlength=nA)
n0 = np.bincount(codes[C == 0], minlength=nA)
e0 = np.bincount(codes[(C == 0) & (F == 1)], minlength=nA)

rng = np.random.default_rng(31)
reps = []
for _ in range(2000):
    idx = rng.integers(0, nA, nA)   # draw authors with replacement, rows follow their author
    a, b, c, d = e1[idx].sum(), n1[idx].sum(), e0[idx].sum(), n0[idx].sum()
    if b and c and d:
        reps.append((a / b) / (c / d))
lo, hi = np.percentile(reps, [2.5, 97.5])
print(f"author-clustered 95% CI   [{lo:.2f}, {hi:.2f}]  ({len(reps)} bootstrap draws)")

entry rate with a seed    9.3722%  (1,784 of 19,035)
entry rate without        1.4841%  (95,035 of 6,403,523)
naive rate ratio          6.32


author-clustered 95% CI   [6.04, 6.59]  (2000 bootstrap draws)


## Step 4: ride against independent, inside the seeded entries

Of the entries that happened with a seed in place, some carry the seed co-author on the entering paper itself and some do not. The split matters because riding along and entering independently are different mechanisms, and only the seeded entries can show it.

In [4]:
rides = int(ev.loc[(C == 1) & (F == 1), "first_entry_ride"].sum())
seeded_entries = int(F[C == 1].sum())
print(f"seeded entries {seeded_entries:,}")
print(f"  with the seed co-author on the entering paper   {rides:,}  ({rides/seeded_entries:.0%})")
print(f"  without, so independent of that person          {seeded_entries-rides:,}  ({1-rides/seeded_entries:.0%})")

seeded entries 1,784
  with the seed co-author on the entering paper   756  (42%)
  without, so independent of that person          1,028  (58%)


## Step 5: the contrast by year

The corpus starts in 2015, so the first years cannot carry seeds, no co-author relation can predate the corpus. The seeded cells only reach a usable size around 2021, everything before rides on a handful of entries and should not be read as a trend.

In [5]:
g = ev.groupby("t", observed=True)
yr = pd.DataFrame({
    "seeded_rows": g.apply(lambda x: int((x.coauthor_seed == 1).sum()), include_groups=False),
    "seeded_entries": g.apply(lambda x: int(((x.coauthor_seed == 1) & (x.F == 1)).sum()), include_groups=False),
    "other_rows": g.apply(lambda x: int((x.coauthor_seed == 0).sum()), include_groups=False),
    "other_entries": g.apply(lambda x: int(((x.coauthor_seed == 0) & (x.F == 1)).sum()), include_groups=False),
})
yr["rate_ratio"] = (yr.seeded_entries / yr.seeded_rows) / (yr.other_entries / yr.other_rows)
print(yr.to_string(float_format=lambda v: f"{v:.2f}"))
print("\nthe cells before 2021 hold single digit seeded entries, their ratios are noise, not a trend")

      seeded_rows  seeded_entries  other_rows  other_entries  rate_ratio
t                                                                       
2015            0               0      117056           1855         NaN
2016            9               5      161085           2461       36.36
2017           46              12      196118           2952       17.33
2018          185              39      326591           4906       14.03
2019          512              77      558551           8475        9.91
2020         1001             141      723699          10631        9.59
2021         1739             170      856449          12734        6.57
2022         3314             343     1052619          15723        6.93
2023         5243             441     1122706          16497        5.72
2024         6986             556     1288649          18801        5.46

the cells before 2021 hold single digit seeded entries, their ratios are noise, not a trend


## Step 6: what T covers, with the reasons named

`topic_match` is the rolling author profile against the journal profile, 0 to 1, from the threshold-free run. `profile_cutoff` sits strictly before t on every filled row, and every empty cell names its reason in `tm_status`

- the whole-period paper threshold is removed, so eligibility no longer looks at future productivity, and journal profiles now build from every embeddable paper
- the remaining gaps are structural, most entrants have no pre-t paper at all, and a smaller block has papers without usable abstracts or journals without a profile before t

In [6]:
has = ev["T"].notna().values
print(f"T exists on {has.sum():,} rows ({has.mean():.1%})")
print(f"  on entry rows            {has[F==1].mean():.1%}")
print(f"  on seeded rows           {has[C==1].mean():.1%}")
print(f"  on seeded entries        {has[(C==1)&(F==1)].mean():.1%}  ({int(has[(C==1)&(F==1)].sum()):,} of {int(((C==1)&(F==1)).sum()):,})")
print(f"  on rides                 {has[ev.first_entry_ride.values==1].mean():.1%}")
print("\nwhy the rest is empty, all rows")
print(ev["tm_status"].value_counts().to_string())
print("\non entry rows")
print(ev.loc[F == 1, "tm_status"].value_counts().to_string())
print("\non seeded entries")
print(ev.loc[(F == 1) & (C == 1), "tm_status"].value_counts().to_string())

T exists on 1,106,356 rows (17.2%)
  on entry rows            12.2%
  on seeded rows           99.2%
  on seeded entries        99.0%  (1,766 of 1,784)
  on rides                 99.6%

why the rest is empty, all rows


tm_status
no_author_history                4989914
ok                               1106356
no_author_and_journal_history     253203
below_threshold_no_abstract        55822
no_journal_history                 17263

on entry rows


tm_status
no_author_history                82281
ok                               11779
no_author_and_journal_history     1931
below_threshold_no_abstract        823
no_journal_history                   5

on seeded entries
tm_status
ok                   1766
no_author_history      18


## Step 7: provisional adjustment, F ~ C + T on the rows where T exists

Provisional for three reasons, the export is a local run of the pipeline pending its owner's approval, the rolling anchor, a co-author may already have pulled the author's topics toward the journal so part of the association is adjusted away, and the complete-case population, which is visibly different, its naive ratio alone shows that. Numbers here are for direction and magnitude, not for the report.

- logistic regression, standard errors clustered by author
- the adjusted ratio comes from predicting every row once with `C = 1` and once with `C = 0` and dividing the mean risks, step 7 to 8 of the planning notebook
- overlap is checked first, adjustment only means something where seeded and unseeded rows share the same range of `T`

In [7]:
m = has
X = np.column_stack([np.ones(m.sum()), C[m].astype(float), ev["T"].values[m]])
y = F[m].astype(float)
print(f"complete cases {int(m.sum()):,} rows, {int(y.sum()):,} entries")
print(f"naive ratio inside this population {(y[X[:,1]==1].mean())/(y[X[:,1]==0].mean()):.2f}, "
      f"against 6.32 in the full table, so the population shift is real\n")

# overlap in T between the two exposure groups
q = [0.05, 0.25, 0.5, 0.75, 0.95]
qt1 = np.quantile(X[X[:, 1] == 1, 2], q)
qt0 = np.quantile(X[X[:, 1] == 0, 2], q)
print("T quantiles 5/25/50/75/95")
print("  with seed    " + "  ".join(f"{v:.2f}" for v in qt1))
print("  without      " + "  ".join(f"{v:.2f}" for v in qt0))
print()

beta = np.zeros(3)
for _ in range(25):   # newton, converges in a handful of steps on this size
    p = 1 / (1 + np.exp(-(X @ beta)))
    H = (X * (p * (1 - p))[:, None]).T @ X
    step = np.linalg.solve(H, X.T @ (y - p))
    beta += step
    if np.abs(step).max() < 1e-10:
        break

# sandwich variance with author clusters
p = 1 / (1 + np.exp(-(X @ beta)))
cl = codes[m]
U = X * (y - p)[:, None]
order = np.argsort(cl)
S = np.zeros((3, 3))
for blk in np.split(U[order], np.flatnonzero(np.diff(cl[order])) + 1):
    s = blk.sum(0)
    S += np.outer(s, s)
Hinv = np.linalg.inv((X * (p * (1 - p))[:, None]).T @ X)
se = np.sqrt(np.diag(Hinv @ S @ Hinv))

print(f"C   log odds {beta[1]:.3f}, OR {np.exp(beta[1]):.2f}, "
      f"cluster 95% CI [{np.exp(beta[1]-1.96*se[1]):.2f}, {np.exp(beta[1]+1.96*se[1]):.2f}]")
print(f"T   log odds {beta[2]:.3f}, per 0.1 of topic fit OR {np.exp(beta[2]/10):.2f}")

X1 = np.column_stack([X[:, 0], np.ones(len(X)), X[:, 2]])
X0 = np.column_stack([X[:, 0], np.zeros(len(X)), X[:, 2]])

def log_rr(b):
    p1 = 1 / (1 + np.exp(-(X1 @ b)))
    p0 = 1 / (1 + np.exp(-(X0 @ b)))
    return np.log(p1.mean() / p0.mean())

# delta method, the ratio is a smooth function of beta so its variance follows from the sandwich
grad = np.zeros(3)
for k in range(3):
    d = np.zeros(3); d[k] = 1e-6
    grad[k] = (log_rr(beta + d) - log_rr(beta - d)) / 2e-6
se_lrr = float(np.sqrt(grad @ (Hinv @ S @ Hinv) @ grad))
rr = np.exp(log_rr(beta))
print(f"\nadjusted rate ratio, g computation  {rr:.2f}, "
      f"cluster 95% CI [{rr*np.exp(-1.96*se_lrr):.2f}, {rr*np.exp(1.96*se_lrr):.2f}]")

complete cases 1,106,356 rows, 11,779 entries
naive ratio inside this population 10.16, against 6.32 in the full table, so the population shift is real

T quantiles 5/25/50/75/95
  with seed    0.73  0.79  0.83  0.88  0.93
  without      0.66  0.72  0.76  0.81  0.88



C   log odds 1.875, OR 6.52, cluster 95% CI [6.14, 6.93]
T   log odds 8.758, per 0.1 of topic fit OR 2.40

adjusted rate ratio, g computation  6.09, cluster 95% CI [5.76, 6.45]


## Step 8: where the change in the adjusted ratio comes from

The thresholded run and the threshold-free run do not measure the same `T`: journal profiles grow with the embedded author set, and the kernel scale re-estimates. To see what moved the adjusted ratio, the same model runs three times in order, first swapping the values on the identical row set, then adding the new rows. This is an ordered diagnostic decomposition, the order matters, so the two shares are indicative rather than a unique attribution.

In [8]:
# the thresholded run, values on the same rows for the value-swap comparison
told = pd.read_csv("../data/event_table_topicmatch_local_min3.csv", usecols=["topic_match"], low_memory=False)["topic_match"].values

def adjusted_rr(mask, T):
    Xd = np.column_stack([np.ones(mask.sum()), C[mask].astype(float), T[mask]])
    yd = F[mask].astype(float)
    b = np.zeros(3)
    for _ in range(30):
        pd_ = 1 / (1 + np.exp(-(Xd @ b)))
        step = np.linalg.solve((Xd * (pd_ * (1 - pd_))[:, None]).T @ Xd, Xd.T @ (yd - pd_))
        b += step
        if np.abs(step).max() < 1e-10:
            break
    p1 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.ones(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    p0 = (1 / (1 + np.exp(-(np.column_stack([Xd[:, 0], np.zeros(len(Xd)), Xd[:, 2]]) @ b)))).mean()
    return p1 / p0

tnew = ev["T"].values
shared = ~np.isnan(told) & ~np.isnan(tnew)
print(f"kernel scale, thresholded 0.3613 against threshold-free 0.3611, so nearly unchanged")
print(f"shared rows {shared.sum():,}")
print(f"  old values, shared rows        {adjusted_rr(shared, told):.2f}")
print(f"  new values, same shared rows   {adjusted_rr(shared, tnew):.2f}")
print(f"  new values, full new rows      {adjusted_rr(~np.isnan(tnew), tnew):.2f}")

kernel scale, thresholded 0.3613 against threshold-free 0.3611, so nearly unchanged
shared rows 623,510


  old values, shared rows        5.24


  new values, same shared rows   5.69


  new values, full new rows      6.09


## Summary

| number | value | status |
|---|---|---|
| crude rate ratio, full table | 6.32, CI 6.04 to 6.59 | solid as a crude association |
| seeded entries | 1,784, of which 756 rides and 1,028 independent | solid, confirmed by the prolog parity check |
| yearly contrast | 5.46 to 6.93 from 2021 | descriptive, no stability claim |
| T coverage | 17.2% of rows, 99.0% of seeded entries | threshold-free local run |
| crude ratio, complete cases | 10.16 | shows the population shift |
| topic-adjusted ratio | 6.09, CI 5.76 to 6.45 | provisional, local threshold-free run |

Removing the whole-period threshold moved the adjusted ratio from 5.24 to 6.09. The ordered decomposition in step 8 puts roughly +0.45 on the changed values, plausibly the journal profiles since the kernel scale barely moved, and +0.40 on the added rows. The order of the decomposition matters, so these shares are a diagnostic, not a unique attribution. Within the threshold-free population the adjustment now changes little against the crude 6.32, and T retains a positive adjusted association with entry (2.40 per 0.1 of topic fit). The independent Prolog comparison matched all event table rows; review of those additional rules remains separate from the Q3 analysis.

Still open, in order

1. Pierre confirms or reruns the threshold-free export himself, the pipeline here reproduces his committed run exactly under its original settings, so this is his method with his proposed change applied
2. the adapter question, the embedding model may be running without its adapter, in the original run and the local one alike, its owner decides whether to keep or change that
3. the group decision on missing `T`, 88% of entries have no pre-t history to build a profile from, observed `T` stays continuous and `tm_status` carries the reason per empty cell
4. the full model of the planning notebook, prior papers, career age, journal breadth, journal and year effects, and the planted-effect calibration, once 1 to 3 are settled